<a href="https://colab.research.google.com/github/lamagesty/analiseDistanciaInternacoes/blob/main/distanciaInternacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 1. IMPORTAÇÕES E PREPARAÇÃO DOS DADOS
# ==========================================

!pip -q install pandas openpyxl requests tqdm rapidfuzz unidecode geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import requests
import time
import numpy as np
from tqdm import tqdm
from rapidfuzz import process, fuzz
from unidecode import unidecode

ARQUIVO = "OrigemDestino_CORE_SUSFÁCIL.xlsx"

core = pd.read_excel(ARQUIVO, sheet_name="CORE")
sus = pd.read_excel(ARQUIVO, sheet_name="SUSFácil")

COL_ORIG = 'Municipio origem'
COL_DEST = 'Municipio destino'
COL_QTD = 'Internacoes'

core = core[
    (~core[COL_ORIG].str.upper().eq("SEM MUNICÍPIO")) &
    (~core[COL_DEST].str.upper().eq("SEM MUNICÍPIO"))
]

sus = sus[
    (~sus[COL_ORIG].str.upper().eq("SEM MUNICÍPIO")) &
    (~sus[COL_DEST].str.upper().eq("SEM MUNICÍPIO"))
]

pares = pd.concat([
    core[[COL_ORIG, COL_DEST]],
    sus[[COL_ORIG, COL_DEST]]
]).drop_duplicates().reset_index(drop=True)

print(f"{len(pares)} pares únicos encontrados.")

2398 pares únicos encontrados.


In [4]:
# ==========================================
# 2. BASE DE MUNICÍPIOS E COORDENADAS
# ==========================================
url_coords = "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"
df_mun_coords = pd.read_csv(url_coords)

# Apenas municípios de Minas Gerais
df_mg = df_mun_coords[df_mun_coords["codigo_uf"] == 31].copy()
df_mg["nome_norm"] = df_mg["nome"].apply(lambda x: unidecode(str(x)).upper())

# Cria a base de municípios existente nos seus dados
base = pd.DataFrame({
    "municipio": pd.concat([
        pares[COL_ORIG],
        pares[COL_DEST]
    ]).drop_duplicates()
})

base["nome_norm"] = base["municipio"].apply(lambda x: unidecode(str(x)).upper())

# Junta coordenadas na base de municípios
base = base.merge(
    df_mg[["nome_norm", "latitude", "longitude"]],
    on="nome_norm",
    how="left"
)

base = base.rename(columns={"latitude": "lat", "longitude": "lon"})

# Descobre qual é o nome atual da coluna
col_lat = [c for c in base.columns if 'lat' in c][0]
col_lon = [c for c in base.columns if 'lon' in c][0]

In [5]:
# ==========================================
# 3. TRAZENDO AS COORDENADAS PARA OS PARES
# ==========================================
# Limpa qualquer coluna antiga de lat/lon do 'pares' para evitar duplicações
colunas_antigas = [c for c in pares.columns if 'lat' in c or 'lon' in c]
pares = pares.drop(columns=colunas_antigas)

# Trazendo as coordenadas para o município de ORIGEM
pares = pares.merge(
    base[['municipio', col_lat, col_lon]],
    left_on=COL_ORIG,
    right_on='municipio',
    how='left'
).rename(columns={col_lat: 'lat_origem', col_lon: 'lon_origem'}).drop(columns=['municipio'])

# Trazendo as coordenadas para o município de DESTINO
pares = pares.merge(
    base[['municipio', col_lat, col_lon]],
    left_on=COL_DEST,
    right_on='municipio',
    how='left'
).rename(columns={col_lat: 'lat_destino', col_lon: 'lon_destino'}).drop(columns=['municipio'])

In [6]:
# ==========================================
# 4. FUNÇÃO DE ROTA (COM SESSÃO E OTIMIZAÇÃO)
# ==========================================
def rota(lat1, lon1, lat2, lon2, sessao=None):
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {"overview": "false"}

    time.sleep(0.5)

    try:
        req = sessao if sessao else requests
        r = req.get(url, params=params)

        if r.status_code != 200:
            return None, None

        dados = r.json()
        if dados.get("code") == "Ok" and len(dados.get("routes", [])) > 0:
            distancia_km = dados['routes'][0]['distance'] / 1000
            duracao_min = dados['routes'][0]['duration'] / 60
            return distancia_km, duracao_min
        else:
            return None, None

    except Exception as e:
        return None, None

In [7]:
# ==========================================
# 5. LOOP DE REQUISIÇÕES
# ==========================================
distancias = []
tempos = []

print(f"\nCalculando rotas para {len(pares)} pares únicos...")

sessao_osrm = requests.Session()

for index, linha in tqdm(pares.iterrows(), total=len(pares)):

    # OTIMIZAÇÃO 1: Origem e Destino iguais
    if linha[COL_ORIG] == linha[COL_DEST]:
        distancias.append(0.0)
        tempos.append(0.0)
        continue

    # OTIMIZAÇÃO 2: Coordenadas ausentes
    if pd.isna(linha["lat_origem"]) or pd.isna(linha["lon_origem"]) or \
       pd.isna(linha["lat_destino"]) or pd.isna(linha["lon_destino"]):
        distancias.append(None)
        tempos.append(None)
        continue

    dist, tempo = rota(
        linha["lat_origem"],
        linha["lon_origem"],
        linha["lat_destino"],
        linha["lon_destino"],
        sessao_osrm
    )
    distancias.append(dist)
    tempos.append(tempo)

pares['distancia_km'] = distancias
pares['tempo_min'] = tempos

print("\nCálculo de rotas finalizado com sucesso!")


Calculando rotas para 2398 pares únicos...


100%|██████████| 2398/2398 [20:53<00:00,  1.91it/s]


Cálculo de rotas finalizado com sucesso!


In [8]:
# ==========================================
# 6. JUNTANDO DISTÂNCIAS E CALCULANDO ESTATÍSTICAS
# ==========================================
core_final = core.merge(
    pares[[COL_ORIG, COL_DEST, 'distancia_km', 'tempo_min', 'lat_origem', 'lon_origem', 'lat_destino', 'lon_destino']],
    on=[COL_ORIG, COL_DEST],
    how='left'
)

sus_final = sus.merge(
    pares[[COL_ORIG, COL_DEST, 'distancia_km', 'tempo_min', 'lat_origem', 'lon_origem', 'lat_destino', 'lon_destino']],
    on=[COL_ORIG, COL_DEST],
    how='left'
)

def calcular_estatisticas(df, nome_modelo):
    df_valido = df.dropna(subset=['distancia_km']).copy()
    distancias_por_paciente = df_valido['distancia_km'].repeat(df_valido['Internacoes'])

    media = distancias_por_paciente.mean()
    mediana = distancias_por_paciente.median()
    maximo = distancias_por_paciente.max()
    soma_km = distancias_por_paciente.sum()
    total_pacientes = len(distancias_por_paciente)

    print(f"\n=== ESTATÍSTICAS: {nome_modelo} ===")
    print(f"Total de Internações (com rota válida): {total_pacientes}")
    print(f"Distância Média (por paciente): {media:.2f} km")
    print(f"Distância Mediana (por paciente): {mediana:.2f} km")
    print(f"Maior Distância percorrida: {maximo:.2f} km")
    print(f"Soma Total de km rodados: {soma_km:,.2f} km")
    print("-" * 40)

    return distancias_por_paciente

print("\nResultados da Análise:")
dist_core = calcular_estatisticas(core_final, "Modelo CORE")
dist_sus = calcular_estatisticas(sus_final, "Modelo SUSFácil")


Resultados da Análise:

=== ESTATÍSTICAS: Modelo CORE ===
Total de Internações (com rota válida): 75866
Distância Média (por paciente): 10.36 km
Distância Mediana (por paciente): 0.00 km
Maior Distância percorrida: 769.63 km
Soma Total de km rodados: 785,618.43 km
----------------------------------------

=== ESTATÍSTICAS: Modelo SUSFácil ===
Total de Internações (com rota válida): 76735
Distância Média (por paciente): 10.31 km
Distância Mediana (por paciente): 0.00 km
Maior Distância percorrida: 817.38 km
Soma Total de km rodados: 791,387.76 km
----------------------------------------


In [9]:
# ==========================================
# 7. EXPORTAÇÃO
# ==========================================
nome_arquivo_core = 'CORE_Final_com_Distancias.xlsx'
nome_arquivo_sus = 'SUSFacil_Final_com_Distancias.xlsx'

core_final.to_excel(nome_arquivo_core, index=False)
sus_final.to_excel(nome_arquivo_sus, index=False)

print("\n✅ Arquivos gerados com sucesso!")

try:
    from google.colab import files
    files.download(nome_arquivo_core)
    files.download(nome_arquivo_sus)
except ImportError:
    pass


✅ Arquivos gerados com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>